# Fine-tuning BERT or XLM-RoBERTa for multi-label text classification - models having quality imformation as loss weights


## Set-up environment

First, we install the libraries which we'll use: HuggingFace Transformers and Datasets.

In [ ]:
!pip install -q datasets

Set the path to get the translation metrics

In [ ]:
import pandas as pd

train_with_extra_data = pd.read_excel('/train_quality_of_translations_both.xlsx')
valid_with_extra_data = pd.read_excel('/valid_quality_of_translations_both.xlsx')
test_with_extra_data = pd.read_excel('/test_quality_of_translations_both.xlsx')

extra_data_df = pd.concat([train_with_extra_data,valid_with_extra_data,test_with_extra_data])
extra_data_df = extra_data_df[['text_id', 'prediction_siamese']]
extra_data_df.sort_values(by = 'text_id')

,text_id,prediction_siamese
0,1,0.913270
0,4,0.682999
1,6,0.451868
0,7,0.894689
1,8,0.873501
...,...,...
4083,7942,0.578753
4084,7943,0.814703
4085,7944,0.827594
4086,7945,0.815839


Maybe you'll need to run this.

In [ ]:
!pip install transformers==4.28.0
!pip install --upgrade accelerate

## Load dataset

We load REDv2_EN / REDv2_EN_ANN, multi-label text classification datasets from the [hub](https://huggingface.co/):

`REDv2_EN` RED translated with Google Translate

`REDv2_EN_ANN` RED translated with Google Translate having the test set reannotated

`REDv2_NLLB` RED translated with NLLB

`REDv2_NLLB_ANN` RED translated with Google Translate having the test set reannotated



In [ ]:
from datasets import load_dataset

dataset = load_dataset("Alegzandra/REDv2_EN_ANN")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


As we can see, the dataset contains 3 splits: one for training, one for validation and one for testing.

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['annotator2', 'agreed_labels', 'Sadness', 'percentage_labels', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'annotator1', 'annotator3', 'sum_labels', 'Fear', 'Anger', 'Surprise'],
        num_rows: 4088
    })
    validation: Dataset({
        features: ['annotator2', 'agreed_labels', 'Sadness', 'percentage_labels', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'annotator1', 'annotator3', 'sum_labels', 'Fear', 'Anger', 'Surprise'],
        num_rows: 543
    })
    test: Dataset({
        features: ['annotator2', 'agreed_labels', 'Sadness', 'percentage_labels', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'annotator1', 'annotator3', 'sum_labels', 'Fear', 'Anger', 'Surprise'],
        num_rows: 818
    })
})

In [ ]:
dataset = dataset.remove_columns(['agreed_labels', 'annotator1', 'annotator2', 'annotator3', 'percentage_labels', 'sum_labels', ])

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Sadness', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise'],
        num_rows: 4088
    })
    validation: Dataset({
        features: ['Sadness', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise'],
        num_rows: 543
    })
    test: Dataset({
        features: ['Sadness', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise'],
        num_rows: 818
    })
})

Add translation metrics as extra_data

In [ ]:
def add_translation_metrics(example):
    prediction_siamese = round(extra_data_df.loc[extra_data_df['text_id'] == example['text_id']]['prediction_siamese'].values.tolist()[0], 8)
    example["extra_data"] = [prediction_siamese]
    return example

In [ ]:
dataset = dataset.map(add_translation_metrics)

Let's check the first example of the training split:

In [ ]:
example = dataset['train'][:5]
print(example['text'])
print(example['text_id'])
print(example['extra_data'])

["Fuck all social networks, they haven't extorted a dime from me since they exist and I use them.", '<|PERSON|> announces that the Victor Babeș Hospital in Timișoara no longer has places for COVID patients: Some of the fellow citizens do not follow any rules, giving credence to fake news and conspiracy theories', 'I thought you were going to kill someone, you scared my wife', "UPDATE. Knife attack in Nice: three dead, several injured. I don't rule out the idea of a terrorist attack...", 'In your eyes I saw an extraordinary talent! #Romanians have talent']
[7, 8, 13, 19, 21]
[[0.89468896], [0.87350082], [0.78655308], [0.90831995], [0.84707046]]


In [ ]:
def prep(in_str):
  elem = in_str.replace("<|EMAIL|>","email")
  elem = elem.replace("<|TEL|>","")
  elem = elem.replace("<|USERNAME|>","")
  elem = elem.replace("<|URL|>","")
  out_str = elem.replace("<|PERSON|>","Person")
  return out_str

print(prep("<|PERSON|> s-a dus la <|EMAIL|> sa <|URL|>"))


Person s-a dus la email sa 


In [ ]:
def add_prefix(example):

    example["text"] = prep(example["text"])

    return example

In [ ]:
dataset = dataset.map(add_prefix)

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Sadness', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise', 'extra_data'],
        num_rows: 4088
    })
    validation: Dataset({
        features: ['Sadness', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise', 'extra_data'],
        num_rows: 543
    })
    test: Dataset({
        features: ['Sadness', 'text', 'Trust', 'text_id', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise', 'extra_data'],
        num_rows: 818
    })
})

In [ ]:
example = dataset['train'][:10]
print(type(example))
print(example['text'])

<class 'dict'>
["Fuck all social networks, they haven't extorted a dime from me since they exist and I use them.", 'Person announces that the Victor Babeș Hospital in Timișoara no longer has places for COVID patients: Some of the fellow citizens do not follow any rules, giving credence to fake news and conspiracy theories', 'I thought you were going to kill someone, you scared my wife', "UPDATE. Knife attack in Nice: three dead, several injured. I don't rule out the idea of a terrorist attack...", 'In your eyes I saw an extraordinary talent! #Romanians have talent', 'So it becomes the entertaining press magazine made through my eyes? 😁', 'Plateau Bella Italia Spend time with your family and leave the cooking to us Restaurant: 📲  delivery safedelivery 🙏 pizza at Bella Italia Romania', 'And why did you get upset with BCR for screwing up your plans... 🤣 well for 30 seconds of action, they knew there was no point in complicating yourself', 'Mine are up to 65 years old, and last week I exas

The dataset consists of tweets, labeled with one or more emotions.

Let's create a list that contains the labels, as well as 2 dictionaries that map labels to integers and back.

In [ ]:
labels = [label for label in dataset['train'].features.keys() if label not in ['text_id', 'text', 'extra_data']]
id2label = {idx:label for idx, label in enumerate(labels)}
label2id = {label:idx for idx, label in enumerate(labels)}
labels

['Sadness', 'Trust', 'Neutral', 'Joy', 'Fear', 'Anger', 'Surprise']

## Preprocess data

As models like BERT don't expect text as direct input, but rather `input_ids`, etc., we tokenize the text using the tokenizer. Here I'm using the `AutoTokenizer` API, which will automatically load the appropriate tokenizer based on the checkpoint on the hub.

What's a bit tricky is that we also need to provide labels to the model. For multi-label text classification, this is a matrix of shape (batch_size, num_labels). Also important: this should be a tensor of floats rather than integers, otherwise PyTorch' `BCEWithLogitsLoss` (which the model will use) will complain, as explained [here](https://discuss.pytorch.org/t/multi-label-binary-classification-result-type-float-cant-be-cast-to-the-desired-output-type-long/117915/3).

**For our experiments the transformer models are used:**

BERT - `bert-base-cased`

XLM RoBERTa - `xlm-roberta-base`

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def preprocess_data(examples):
  # take a batch of texts
  text = examples["text"]
  # encode them
  encoding = tokenizer(text, padding="max_length", truncation=True, max_length=128)
  # add labels
  labels_batch = {k: examples[k] for k in examples.keys() if k in labels}
  # create numpy array of shape (batch_size, num_labels)
  labels_matrix = np.zeros((len(text), len(labels)))
  # fill numpy array
  for idx, label in enumerate(labels):
    labels_matrix[:, idx] = labels_batch[label]

  encoding["labels"] = labels_matrix.tolist()

  return encoding

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
columns_to_remove = dataset['train'].column_names
columns_to_remove.remove('extra_data')

encoded_dataset = dataset.map(preprocess_data, batched=True, remove_columns=columns_to_remove)

Map:   0%|          | 0/543 [00:00<?, ? examples/s]

In [ ]:
example = encoded_dataset['train'][0]
print(example.keys())

dict_keys(['extra_data', 'input_ids', 'attention_mask', 'labels'])


In [ ]:
tokenizer.decode(example['input_ids'])

"<s> Fuck all social networks, they haven't extorted a dime from me since they exist and I use them.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>"

In [ ]:
example['labels']

[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]

In [ ]:
[id2label[idx] for idx, label in enumerate(example['labels']) if label == 1.0]

['Anger']

Finally, we set the format of our data to PyTorch tensors. This will turn the training, validation and test sets into standard PyTorch [datasets](https://pytorch.org/docs/stable/data.html).

In [ ]:
encoded_dataset.set_format("torch")

In [ ]:
type(encoded_dataset)

datasets.dataset_dict.DatasetDict

In [ ]:
encoded_dataset.keys()

dict_keys(['train', 'validation', 'test'])

In [ ]:
encoded_dataset["train"][0]

{'extra_data': tensor([0.8947]),
 'input_ids': tensor([    0, 88046,   756,  2265, 33120,     7,     4,  1836, 38246,    25,
            18,  1119,  1290,  3674,    10,    45,   282,  1295,   163, 16792,
          1836, 32316,   136,    87,  4527,  2856,     5,     2,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
      

In [ ]:
encoded_dataset["train"][100]

{'extra_data': tensor([0.5688]),
 'input_ids': tensor([     0,    337,     31,    297,    390,     70,    881,   1314,   1295,
          32783,     11,   3291,    941,      5,    581,  63805,    450, 143434,
         139949,   1363,   8305,   4143,   1510,    202,   1620,  55283,  45803,
             99,     70,    160,  30319,  29398,     23,  32783,     11,   3291,
            941,      5,  19614,   9167,      9, 150621,   2037,     38,      2,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1

## Define model

Here we define a model that includes a pre-trained base (i.e. the weights from bert-base-uncased) are loaded, with a random initialized classification head (linear layer) on top. One should fine-tune this head, together with the pre-trained base on a labeled dataset.

This is also printed by the warning.

We set the `problem_type` to be "multi_label_classification", as this will make sure the appropriate loss function is used (namely [`BCEWithLogitsLoss`](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)). We also make sure the output layer has `len(labels)` output neurons, and we set the id2label and label2id mappings.

-----------------------------------------------------

Change `XLMRobertaForSequenceClassification` into `BertForSequenceClassification` and `self.bert =  BertModel(config)` when working with BERT model.

**For our experiments transformer models are chosen using the following schema:**

Models 11 and 13 - BERT

Models 12 and 14 - XLM-RoBERTa

In [ ]:
import torch
from torch import nn
from transformers import XLMRobertaConfig, XLMRobertaModel, XLMRobertaForSequenceClassification
from transformers.modeling_outputs import SequenceClassifierOutput
from typing import Optional, Union, Tuple

class CustomSequenceClassification(XLMRobertaForSequenceClassification):

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        # original model
        self.roberta =  XLMRobertaModel(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        # Initialize weights and apply final processing
        self.post_init()


    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        extra_data: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, SequenceClassifierOutput]:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs[1]

        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = nn.MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                #### adding sample weights using prediction siamese
                if self.training:
                    loss_fct = nn.BCEWithLogitsLoss(reduce=False)
                    loss = loss_fct(logits, labels)
                    loss = loss.mean(dim = 1)
                    loss = loss * extra_data.view(-1)
                    loss = loss.mean(dim = 0)
                else:
                    loss_fct = nn.BCEWithLogitsLoss()
                    loss = loss_fct(logits, labels)
                ####


        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

**For our experiments the transformer models are used:**

BERT - `bert-base-cased`

XLM RoBERTa - `xlm-roberta-base`

In [ ]:
new_model = CustomSequenceClassification.from_pretrained("xlm-roberta-base",
                                                         num_labels=len(labels),
                                                         id2label=id2label,
                                                         label2id=label2id,
                                                         problem_type='multi_label_classification')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at xlm-roberta-base were not used when initializing CustomSequenceClassification: ['lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias']
- This IS expected if you are initializing CustomSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassifica

## Train the model!

We are going to train the model using HuggingFace's Trainer API. This requires us to define 2 things:

* `TrainingArguments`, which specify training hyperparameters. All options can be found in the [docs](https://huggingface.co/transformers/main_classes/trainer.html#trainingarguments). Below, we for example specify that we want to evaluate after every epoch of training, we would like to save the model every epoch, we set the learning rate, the batch size to use for training/evaluation, how many epochs to train for, and so on.
* a `Trainer` object (docs can be found [here](https://huggingface.co/transformers/main_classes/trainer.html#id1)).

In [ ]:
batch_size = 8
metric_name = "f1"

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    f"xlm-roberta-base-finetuned-on-REDv2_EN_ANN_2",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    gradient_accumulation_steps = 2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    push_to_hub=False,
)



We are also going to compute metrics while training. For this, we need to define a `compute_metrics` function, that returns a dictionary with the desired metric values.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch

def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions,
            tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds,
        labels=p.label_ids)
    return result

Let's start training!

In [ ]:
trainer = Trainer(
    new_model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.10/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


Epoch,Training Loss,Validation Loss,F1,Roc Auc,Accuracy
0,No log,0.327540,0.435780,0.644154,0.300184
2,0.269000,0.267946,0.628837,0.756245,0.515654
2,0.269000,0.268130,0.643432,0.770739,0.535912
4,0.169100,0.258261,0.666667,0.785390,0.567219
4,0.169100,0.262718,0.686491,0.804433,0.587477
6,0.122400,0.277142,0.648927,0.781030,0.563536
6,0.122400,0.281499,0.671769,0.795837,0.578269
8,0.091200,0.295209,0.649063,0.782536,0.556169
8,0.091200,0.296273,0.660441,0.789688,0.565378
9,0.073200,0.298365,0.663858,0.793139,0.561694


/usr/local/lib/python3.10/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))
/usr/local/lib/python3.10/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))
/usr/local/lib/python3.10/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))
/usr/local/lib/python3.10/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))
/usr/local/lib/python3.10/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warnin

TrainOutput(global_step=2550, training_loss=0.14348686293059704, metrics={'train_runtime': 1655.1622, 'train_samples_per_second': 24.698, 'train_steps_per_second': 1.541, 'total_flos': 2683853245440000.0, 'train_loss': 0.14348686293059704, 'epoch': 9.98})

## Evaluate

After training, we evaluate our model on the test set.

In [ ]:
# Save the model
#trainer.save_model()

In [ ]:
trainer.evaluate(eval_dataset=encoded_dataset["test"])

{'eval_loss': 0.33465778827667236,
 'eval_f1': 0.6438569206842923,
 'eval_roc_auc': 0.7644594679977254,
 'eval_accuracy': 0.4474327628361858,
 'eval_runtime': 5.8329,
 'eval_samples_per_second': 140.239,
 'eval_steps_per_second': 17.658,
 'epoch': 9.98}

Evaluating on re-annotated english texts